# Данный ноутбук сохраняет все обученные модели в файлы, для удобства из использования

In [ ]:
%%writefile DenseNet_model.py

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

# 1. Определяем Dataset
class ChestXrayDataset(Dataset):
    def __init__(self, csv_file, disease_columns, transform=None):
        self.data = pd.read_csv(csv_file)
        self.disease_columns = disease_columns
        self.transform = transform
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_path = self.data.iloc[idx]['full_path']
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        labels = self.data.iloc[idx][self.disease_columns].values
        labels = torch.tensor(labels.astype(np.float32))
        
        return image, labels

# 2. Определяем diseases_list
diseases_list = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 
                 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration',
                 'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening',
                 'Pneumonia', 'Pneumothorax']

# 3. Трансформации для теста
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 4. Создаем val_loader из тестовой выборки (файл test_data.csv)
print("📁 Загружаем тестовые данные...")
val_dataset = ChestXrayDataset(
    csv_file='3//test_data.csv',  # файл с тестовой выборкой
    disease_columns=diseases_list,
    transform=val_transform
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(f"✅ Тестовых данных: {len(val_dataset)} изображений")

# 5. Загружаем модель если еще не загружена
print("\n🧠 Загружаем модель...")
import torchvision.models.densenet
torch.serialization.add_safe_globals([torchvision.models.densenet.DenseNet])

# Укажи правильный путь к файлу модели
model_path = 'full_model_dn_121.pth'  # или 'best_model.pth'
model = torch.load(model_path, weights_only=False)

# 6. Определяем устройство
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"📱 Устройство: {device}")

# 7. Перемещаем модель на устройство
model = model.to(device)
model.eval()

print("✅ Модель готова к тестированию")

# 8. Теперь можно запустить код для тестирования
print("\n" + "=" * 50)
print("🧪 НАЧИНАЕМ ТЕСТИРОВАНИЕ МОДЕЛИ")
print("=" * 50)

# сохранение в файл модели EfficientNetV2-S

In [1]:
%%writefile EfficientNetV2_S_model.py

import torch
import torch.nn as nn
from torchvision import models, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

diseases_list = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 
                 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration',
                 'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening',
                 'Pneumonia', 'Pneumothorax']

def get_efficientnetv2_s(num_classes=15):
    model = models.efficientnet_v2_s(pretrained=False)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.4, inplace=True),
        nn.Linear(in_features, 768),
        nn.BatchNorm1d(768),
        nn.SiLU(inplace=True),
        nn.Dropout(0.2, inplace=True),
        nn.Linear(768, 384),
        nn.BatchNorm1d(384),
        nn.SiLU(inplace=True),
        nn.Dropout(0.133, inplace=True),
        nn.Linear(384, num_classes)
    )
    return model

model = get_efficientnetv2_s(num_classes=len(diseases_list))
checkpoint = torch.load('efficientnetv2_s_final.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

val_transform = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print(f"Модель загружена на {device}")

Writing EfficientNetV2_S_model.py


# сохранение в файл модели swin_tiny

In [3]:
import torch

#  Проверка что модель корректно сохранилась и загружается
print("\n" + "=" * 60)
print(" ПРОВЕРКА СОХРАНЕННОЙ МОДЕЛИ")
print("=" * 60)

# Загружаем модель для проверки
test_checkpoint = torch.load('swin_tiny_best.pth', map_location=DEVICE)
test_model = get_swin_tiny(num_classes=len(diseases_list))
test_model.load_state_dict(test_checkpoint['model_state_dict'])
test_model = test_model.to(DEVICE)
test_model.eval()

print(f" Модель успешно загружена:")
print(f"   • Файл: swin_tiny_best.pth")
print(f"   • Val Loss при сохранении: {test_checkpoint['val_loss']:.4f}")
print(f"   • Эпоха: {test_checkpoint['epoch']}")
print(f"   • Размер изображения: {test_checkpoint['image_size']}")
print(f"   • Loss config: gamma={test_checkpoint['loss_config']['gamma']}, alpha={test_checkpoint['loss_config']['alpha']}")


 ПРОВЕРКА СОХРАНЕННОЙ МОДЕЛИ


NameError: name 'DEVICE' is not defined

In [4]:
import numpy as np
from sklearn.metrics import roc_auc_score

# Определяем устройство
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Устройство: {device}")

# diseases_list должен быть определен
if 'diseases_list' not in locals():
    diseases_list = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 
                     'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration',
                     'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening',
                     'Pneumonia', 'Pneumothorax']

# 1. Загружаем обученную модель Swin-Tiny
def get_swin_tiny(num_classes=15, dropout_rate=0.4):
    import torch.nn as nn
    from torchvision import models
    
    model = models.swin_t(weights=None)  # Без предобученных весов
    
    in_features = model.head.in_features
    model.head = nn.Sequential(
        nn.Dropout(dropout_rate, inplace=True),
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.GELU(),
        nn.Dropout(dropout_rate/2, inplace=True),
        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.GELU(),
        nn.Dropout(dropout_rate/3, inplace=True),
        nn.Linear(256, num_classes)
    )
    return model

# Создаем модель и загружаем веса
model = get_swin_tiny(num_classes=len(diseases_list))
checkpoint = torch.load('swin_tiny_best.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

# Убедись, что модель на нужном устройстве и в eval режиме
model = model.to(device)
model.eval()
print(f" Модель Swin-Tiny загружена (val_loss: {checkpoint['val_loss']:.4f})")

 Устройство: cuda
 Модель Swin-Tiny загружена (val_loss: 0.0049)


In [7]:
%%writefile swin_tiny_model.py

import torch
import torch.nn as nn
from torchvision import models, transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

diseases_list = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 
                 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration',
                 'Mass', 'No Finding', 'Nodule', 'Pleural_Thickening',
                 'Pneumonia', 'Pneumothorax']

def get_swin_tiny(num_classes=15, dropout_rate=0.4):
    model = models.swin_t(weights=None)
    
    in_features = model.head.in_features
    model.head = nn.Sequential(
        nn.Dropout(dropout_rate, inplace=True),
        nn.Linear(in_features, 512),
        nn.BatchNorm1d(512),
        nn.GELU(),
        nn.Dropout(dropout_rate/2, inplace=True),
        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.GELU(),
        nn.Dropout(dropout_rate/3, inplace=True),
        nn.Linear(256, num_classes)
    )
    return model

# Создаем модель и загружаем веса
model = get_swin_tiny(num_classes=len(diseases_list))
checkpoint = torch.load('swin_tiny_best.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])

# Перемещаем модель на устройство и переводим в eval режим
model = model.to(device)
model.eval()

# Трансформации для Swin-Tiny (из checkpoint)
image_size = checkpoint.get('image_size', 512)  # Получаем размер из checkpoint
val_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

from PIL import Image
import numpy as np
from typing import List

def get_logits_from_image(image_path: str) -> np.ndarray:
    image = Image.open(image_path).convert('RGB')
    image_tensor = val_transform(image).unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits = model(image_tensor)
    
    return logits.cpu().numpy()

def get_probabilities_from_image(image_path: str) -> np.ndarray:
    logits = get_logits_from_image(image_path)
    return 1 / (1 + np.exp(-logits))

def get_logits_batch(image_paths: List[str], batch_size: int = 8) -> np.ndarray:
    all_logits = []
    
    for i in range(0, len(image_paths), batch_size):
        batch_paths = image_paths[i:i+batch_size]
        batch_images = []
        
        for path in batch_paths:
            image = Image.open(path).convert('RGB')
            image_tensor = val_transform(image)
            batch_images.append(image_tensor)
        
        batch_tensor = torch.stack(batch_images).to(device)
        
        with torch.no_grad():
            batch_logits = model(batch_tensor)
        
        all_logits.append(batch_logits.cpu().numpy())
    
    return np.vstack(all_logits) if all_logits else np.array([])

print(f"Модель Swin-Tiny загружена на {device}")
print(f"Размер изображения: {image_size}x{image_size}")
print(f"Val Loss: {checkpoint.get('val_loss', 'N/A')}")



Writing swin_tiny_model.py
